In [1]:
import os
from pyspark.sql import SparkSession
ss=SparkSession.builder.appName('app_for_struct_stream').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 19:25:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
ss.stop()

26/08/29 17:41:53 WARN FileStreamSource: Listed 1 file(s) in 3006 ms
26/08/29 17:42:05 WARN FileStreamSource: Listed 1 file(s) in 2604 ms
26/08/29 17:46:02 WARN FileStreamSource: Listed 1 file(s) in 81306 ms
26/08/29 17:57:19 WARN FileStreamSource: Listed 1 file(s) in 86988 ms
26/08/29 18:03:23 WARN MicroBatchExecutionContext: Query progress update takes longer than batch processing time. Progress update takes 139034 milliseconds. Batch processing takes 123962 milliseconds


In [4]:
from pyspark.sql.types import *
# 1. Define the schema in the begining to have read stream work on empty folder
user_schema = StructType([
                            StructField("empid", IntegerType(), True),
                            StructField("login_attempt_time", TimestampType(), True),
                            StructField("success_or_failure", StringType(), True),
                            StructField("file_batch_id", IntegerType(), True)
                        ])

In [5]:
ss.conf.set("spark.sql.streaming.schemaInference", True)
read_stream_df =    ss.readStream.format("csv")\
                    .options(header = True,delimiter = ",", recursiveFileLookup = True )\
                    .schema(user_schema)\
                    .load("file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_input_folder/")

In [6]:
from pyspark.sql.functions import udf
from datetime import datetime,timedelta
from pyspark.sql.types import *

@udf
def func_date_time():
    return str(datetime.now())

transform_df = read_stream_df.withColumn("Read_time",func_date_time().cast(TimestampType()))

/home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/lib/python3.12/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [7]:
from pyspark.sql.functions import *
transform_df_change_type = transform_df.withColumn("empid", col("empid").cast(IntegerType()))\
                                        .withColumn("login_attempt_time", col("login_attempt_time").cast(TimestampType()))\
                                        .withColumn("success_or_failure", col("success_or_failure").cast(StringType()))\
                                        .withColumn("file_batch_id", col("file_batch_id").cast(IntegerType()))

transform_df_change_type.printSchema()

root
 |-- empid: integer (nullable = true)
 |-- login_attempt_time: timestamp (nullable = true)
 |-- success_or_failure: string (nullable = true)
 |-- file_batch_id: integer (nullable = true)
 |-- Read_time: timestamp (nullable = true)



In [8]:
writing_df_append =    transform_df_change_type.writeStream\
                .format("csv")\
                .options(header = True,delimiter = ",")\
                .option("path", "file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_output_folder")\
                .option("checkpointLocation","file:///home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/y_data_related_folder/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()

26/08/20 19:27:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yv05/Documents/y_claude_code_folder/y_spark_related/y_env_for_struct_stream/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
writing_df_append =    transform_df_change_type.withWatermark("my_time_type_tmstmp", "0 seconds")\
                .groupBy(
                    window(col("my_time_type_tmstmp"), "10 seconds"),
                    col("col_company")
                ) \
                .agg(count("*").alias("event_count")) \
                .writeStream\
                .format("parquet")\
                .options(header = True,delimiter = ",")\
                .option("path", "file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_output_of_ss/")\
                .option("checkpointLocation","file://"+os.environ['my_home_directory']+"/Documents/y_compendium/y_youtube_teaching/y_pyspark/y_datafiles/y_checkpointing_location/")\
                .outputMode("append")\
                .start()
writing_df_append.awaitTermination()